In [ ]:
!pip -q install openai-whisper gTTS jiwer
!apt-get -qq update
!apt-get -qq install -y ffmpeg

In [ ]:
import os
import time
import whisper
from gtts import gTTS
from jiwer import wer
from google.colab import files
from IPython.display import Audio, display

In [ ]:
model = whisper.load_model("base")
print("Whisper model loaded successfully.")

In [ ]:
uploaded = files.upload()
audio_file = next(iter(uploaded.keys()))
print("Input audio file:", audio_file)

In [ ]:
start_time = time.time()
result = model.transcribe(audio_file)
stt_time = time.time() - start_time

recognized_text = result["text"].strip()
detected_language = result.get("language", "unknown")

print("Input file:", audio_file)
print("Recognized text:", recognized_text)
print("Detected language:", detected_language)
print(f"Processing time: {stt_time:.2f} seconds")

display(Audio(filename=audio_file))

In [ ]:
reference_text = input("Enter the correct reference transcript: ").strip()

if reference_text and recognized_text:
    error_rate = wer(reference_text.lower(), recognized_text.lower())
    print("Reference transcript:", reference_text)
    print("Recognized transcript:", recognized_text)
    print(f"Word Error Rate (WER): {error_rate:.4f}")
    print(f"Word Error Rate (percentage): {error_rate * 100:.2f}%")
    if error_rate == 0:
        print("Result: Exact word match.")
    elif error_rate <= 1:
        print("Result: Transcription contains word errors.")
    else:
        print("Result: High error rate; check that the reference transcript matches the audio.")
else:
    error_rate = None
    print("Reference or recognized text is empty; evaluation skipped.")

In [ ]:
if recognized_text:
    tts = gTTS(text=recognized_text, lang=detected_language)
    output_audio = "/content/generated_speech.mp3"
    tts.save(output_audio)
    print("Generated speech output:")
    display(Audio(filename=output_audio))
    files.download(output_audio)
else:
    output_audio = None
    print("No recognized text is available for speech generation.")

In [ ]:
print("Input audio:", audio_file)
print("Detected language:", detected_language)
print("Recognized text:", recognized_text)
print(f"STT processing time: {stt_time:.2f} seconds")

if reference_text.strip() and recognized_text.strip():
    final_wer = wer(reference_text.lower(), recognized_text.lower())
    print(f"Word Error Rate: {final_wer:.2%}")
else:
    final_wer = None
    print("Word Error Rate: Reference transcript required.")

if output_audio and os.path.exists(output_audio):
    print("TTS output: Generated successfully")
    print("Audio file: generated_speech.mp3")
else:
    print("TTS output: Not completed")

In [ ]:
print(f"STT Processing Time: {stt_time:.2f} seconds")

if reference_text.strip() and recognized_text.strip():
    error_rate = wer(reference_text.lower(), recognized_text.lower())
    print(f"Word Error Rate (WER): {error_rate * 100:.2f}%")
    if 0 <= error_rate <= 1:
        print(f"Approximate Word Accuracy: {(1 - error_rate) * 100:.2f}%")
    else:
        print("Word Accuracy: Not reported because WER exceeds 100%.")
else:
    print("WER: Reference transcript required.")

if output_audio and os.path.exists(output_audio):
    print("TTS Generation: Successful")
    print("Output Format: MP3")
else:
    print("TTS Generation: Not completed")